In [ ]:
#pip install atproto numpy pandas matplotlib

# Social Media and Network Analytics - Assignment 2
## Our Question:
- What are peoples limits to Ai generated content in video games?
## Team Members:
- Mitchell Long: S4007880
- Jimmy Vu: S4006701
- Khang Nguyen: S4005711

In [ ]:
from atproto import Client
from account import getAccount
import json

# Running this block will prompt user input for username and password
username, password = getAccount()
client = Client()
client.login(username, password)

### Data Collection and Formatting

In [ ]:
# Run a search query and collect posts
search_results = client.app.bsky.feed.search_posts({"q": "Artificial Intelligence", "q": "games", "limit": 100})
display(search_results)
structured_posts = []

# Iterate through each collected post and its replies
for post in search_results.posts:

    # Print posts for output proof
    print(post.author.display_name + ": ")
    print(post.record.text)
    print('\n')

    # Fetch thread for this post using the uri
    thread = client.app.bsky.feed.get_post_thread({
        "uri": post.uri
    })

    replies_data = []

    replies = getattr(thread.thread, "replies", [])

    # Create JSON structure for each reply
    for reply in replies:
        if hasattr(reply, "post"):

            reply_post = reply.post

            replies_data.append({
                "reply_id": reply_post.uri,
                "text": reply_post.record.text,
                "created_at": getattr(reply_post.record, "created_at", None),

                "author": {
                    "display_name": reply_post.author.display_name,
                    "handle": reply_post.author.handle
                },

                "engagement": {
                    "likes": reply_post.like_count,
                    "replies": reply_post.reply_count,
                    "reposts": reply_post.repost_count
                }
            })

    # Create JSON structure for original post and append comments
    structured_post = {
        "post_id": post.uri,
        "text": post.record.text,
        "created_at": getattr(post.record, "created_at", None),

        "author": {
            "display_name": post.author.display_name,
            "handle": post.author.handle,
            "did": post.author.did
        },

        "engagement": {
            "likes": post.like_count,
            "reposts": post.repost_count,
            "replies": post.reply_count,
            "quotes": post.quote_count
        },

        "metadata": {
            "lang": getattr(post.record, "langs", [])
        },

        # Comments appended here
        "comments": replies_data
    }

    # Add posts to one object
    structured_posts.append(structured_post)

# Create and save data to JSON file
data = {'posts': structured_posts}

with open("bluesky_posts.json", "w", encoding="utf-8") as file:
    json.dump(data, file, indent=4, ensure_ascii=False)

Resort the json by likes so we can collect network of the most influential posts

In [ ]:
with open("bluesky_posts.json", "r") as file:
    data = json.load(file)

data["posts"].sort(
    key=lambda post: post["engagement"]["likes"],
    reverse=True
)

with open("bluesky_posts.json", "w", encoding="utf-8") as file:
    json.dump(data, file, indent=4, ensure_ascii=False)